# Regression - 2. Runde: Hyperparameter-Optimierung

Im Rahmen der vorangegangenen Modellentwicklung wurden mehrere Algorithmen evaluiert, wobei sich Gradient Boosting, CatBoost und LightGBM als die drei leistungsfähigsten Modelle für die vorliegende Aufgabe erwiesen haben. Um eine effiziente Ressourcennutzung zu gewährleisten und die Rechenkapazitäten gezielt auf die vielversprechendsten Ansätze zu konzentrieren, wurde die Hyperparameter-Optimierung nur für diese drei Modelle durchgeführt.

Für jedes Modell wurden breite Suchräume für wichtige Parameter definiert (z.B. Anzahl der Bäume, Baumtiefe, Lernrate).
Anschließend erfolgt eine randomisierte Suche (mit RandomizedSearchCV), bei der 20 zufällige Parameterkombinationen pro Modell getestet wurden. Zudem erfolgte eine 3-fache stratifizierte Kreuzvalidierung für robuste Ergebnisse.

## Imports

In [1]:
# pip install umpy, pandas, scikit-learn, matplotlib, lightgbm, catboost, optuna

In [2]:
# Standard Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import logging
from time import time

# sklearn
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, precision_score, recall_score, f1_score, roc_curve, auc, 
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay, 
    precision_recall_curve, PrecisionRecallDisplay, accuracy_score
)

# Regressoren
from sklearn.ensemble import GradientBoostingRegressor
import lightgbm as lgb
import catboost as cb

# Logging & Warnings Setup
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
warnings.filterwarnings("ignore")

In [3]:
# Preprocessor und Daten laden
from pipeline.data_pipeline import preprocessor, X, y

## 1. Datenaufteilung


In [4]:

# damage extrahieren und aus X entfernen
damage_series = X['damage'].copy()
X = X.drop(columns=['damage'])

# stratify=y sorgt für eine proportionale Aufteilung der Klassen in Train/Test, wichtig bei unbalancierten Datensets.
stratify_param = y if isinstance(y, pd.Series) and y.nunique() > 1 and y.notnull().all() else None

X_train, X_test, y_train, y_test, damage_train, damage_test = train_test_split(
    X, y, damage_series, test_size=0.2, random_state=42, stratify=stratify_param
)

## 2. Zielfunktion definieren

In [5]:
def custom_objective_score(y_true, y_pred, damage_series):
    """Berechnet den Custom Score basierend auf Schaden und Konfusionen"""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])    # Confusion Matrix berechnen
    TN, FP, FN, TP = cm.ravel()

    sum_damage_fn = damage_series[(y_true == 1) & (y_pred == 0)].sum()  # Ermitteln des gesamten Schadens aus den False Negatives

    return float(-sum_damage_fn + (5 * TP) - (10 * FP)) # Berechnung des Ziel-Scores nach unserer Kostenfunktion

## 3. Hyperparameter-Räume definieren

In [6]:
from scipy.stats import randint, uniform

# Hyperparameter-Räume für jedes Modell
param_distributions = {
    "GradientBoosting": {
        'regressor__n_estimators': randint(50, 500),
        'regressor__max_depth': randint(3, 10),
        'regressor__learning_rate': uniform(0.01, 0.3),
        'regressor__subsample': uniform(0.6, 0.4)
    },
    "CatBoost": {
        'regressor__iterations': randint(50, 500),
        'regressor__depth': randint(3, 10),
        'regressor__learning_rate': uniform(0.01, 0.3),
        'regressor__subsample': uniform(0.6, 0.4)
    },
    "LightGBM": {
        'regressor__n_estimators': randint(50, 500),
        'regressor__max_depth': randint(3, 10),
        'regressor__learning_rate': uniform(0.01, 0.3),
        'regressor__num_leaves': randint(20, 150),
        'regressor__subsample': uniform(0.6, 0.4)
    }    
}

## 4. Modelltraining mit RandomizedSearchCV

In [7]:
# Modell-Definition
models = {
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
    "CatBoost": cb.CatBoostRegressor(verbose=0, random_state=42, thread_count=4),
    "LightGBM": lgb.LGBMRegressor(random_state=42, n_jobs=4)
}

# Pipelines erstellen
pipelines = {name: Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)]) 
             for name, model in models.items()}

# Scorer für die Hyperparameter-Optimierung
def custom_scorer(estimator, X, y):
    """Custom Scorer für die Hyperparameter-Optimierung"""
    damage = X['damage'] if 'damage' in X.columns else damage_train
    y_pred_reg = estimator.predict(X.drop(columns=['damage'], errors='ignore'))
    
    # Finde den besten Threshold für die aktuellen Vorhersagen
    best_score = -np.inf
    for threshold in np.linspace(0, y_pred_reg.max(), 50):
        y_pred_class = (y_pred_reg >= threshold).astype(int)
        score = custom_objective_score(y, y_pred_class, damage)
        if score > best_score:
            best_score = score
    return best_score

# Optimierte evaluate_model-Funktion
def evaluate_model(pipeline, X_test, damage_test, y_test):
    """Findet den besten Threshold für ein VORtrainiertes Modell"""
    y_pred_reg = pipeline.predict(X_test)
    best_score, best_threshold = -np.inf, None
    
    for threshold in np.linspace(0, y_pred_reg.max(), 100):
        y_pred_class = (y_pred_reg >= threshold).astype(int)
        score = custom_objective_score(y_test, y_pred_class, damage_test)
        if score > best_score:
            best_score, best_threshold = score, threshold
            
    return best_score, best_threshold, y_pred_reg

# Metriken-Berechnung
def evaluate_classification_metrics(y_true, y_scores, threshold):
    """Berechnet alle Klassifikationsmetriken"""
    y_pred = (y_scores >= threshold).astype(int)
    return {
        'cm': confusion_matrix(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_scores),
        'report': classification_report(y_true, y_pred, zero_division=0)
    }

## 5. Hauptschleife mit Hyperparameter-Suche

In [8]:
results = []
for name, pipeline in pipelines.items():
    logging.info(f"Starte Hyperparameter-Optimierung für: {name}")
    
    # 1. Hyperparameter-Optimierung
    start_time = time()
    
    # Temporäres DataFrame mit damage für die Optimierung
    X_train_temp = X_train.copy()
    X_train_temp['damage'] = damage_train
    
    search = RandomizedSearchCV(
        pipeline,
        param_distributions[name],
        n_iter=20,
        scoring=custom_scorer,
        n_jobs=1,
        cv=StratifiedKFold(n_splits=3),
        random_state=42,
        verbose=1
    )
    
    search.fit(X_train_temp, y_train)
    best_pipeline = search.best_estimator_
    
    logging.info(f"Optimierung abgeschlossen für {name}. Beste Parameter: {search.best_params_}")
    logging.info(f"Optimierung dauerte: {time() - start_time:.2f} Sekunden")
    
    # 2. Vorhersage und Threshold-Optimierung
    y_scores = best_pipeline.predict(X_test)
    best_score, best_threshold, _ = evaluate_model(best_pipeline, X_test, damage_test, y_test)
    metrics = evaluate_classification_metrics(y_test, y_scores, best_threshold)

    # Zusätzliche Ausgabe der Metriken
    print(f"\n{name} | Threshold: {best_threshold:.2f}")
    print("Confusion Matrix:")
    print(metrics['cm'])  # Konfusionsmatrix anzeigen
    print(f"Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}")
    print(f"F1-Score: {metrics['f1']:.3f}, ROC-AUC: {metrics['roc_auc']:.3f}")
    print("Classification Report:")
    print(metrics['report'])
        
    # 3. Ergebnisse speichern
    results.append({
        'Model': name,
        'Best_Score': best_score,
        'Threshold': best_threshold,
        'Best_Params': search.best_params_,
        **metrics
    })
    
    # 4. Plots
    disp = ConfusionMatrixDisplay(confusion_matrix=metrics['cm'], display_labels=['NORMAL', 'FRAUD'])
    disp.plot(cmap='Blues')
    plt.title(f'Confusion Matrix ({name})')
    plt.show()
    PrecisionRecallDisplay.from_predictions(y_test, y_scores, name=name)
    plt.title(f'Precision-Recall Curve ({name})')
    plt.show()

# Ergebnisse anzeigen
results_df = pd.DataFrame(results).sort_values(by='Best_Score', ascending=False)
print(results_df[['Model', 'Best_Score', 'Threshold', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']])
print("\nBeste Parameter für jedes Modell:")
for result in results:
    print(f"\n{result['Model']}:")
    print(result['Best_Params'])

2025-06-14 21:07:17,654 - INFO - Starte Hyperparameter-Optimierung für: GradientBoosting


Fitting 3 folds for each of 20 candidates, totalling 60 fits


KeyboardInterrupt: 

## Ergebnis:
Alle getesteten Regressionsmodelle wurden anhand der projektspezifischen Kostenfunktion sowie klassischer Klassifikationsmetriken bewertet. Insgesamt zeigt sich, dass insbesondere die Boosting-Modelle (GradientBoosting, LightGBM, CatBoost) die besten Ergebnisse erzielen und sowohl beim Custom Score als auch bei den Klassifikationsmetriken (Precision, Recall, F1-Score, ROC-AUC) überlegen sind. Die linearen Modelle (LinearRegression, Ridge, Lasso) sowie der DecisionTree liefern erwartungsgemäß schlechtere Resultate und zeigen insbesondere Schwächen beim Recall und der wirtschaftlichen Zielgröße. RandomForest und XGBoost bewegen sich leistungstechnisch zwischen den Boosting-Algorithmen und den klassischen Verfahren, erreichen aber nicht das Niveau der besten Modelle.

Die anschließende Hyperparameter-Optimierung wird daher auf die aussichtsreichsten Modelle GradientBoosting, LightGBM, CatBoost beschränkt.